# Shopper's Mind Segmentation Challenge - Solution

This notebook implements the solution for the Shopper's Mind Segmentation Challenge. 
It includes data generation (mock), preprocessing, clustering (K-Means), and submission file generation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import os

%matplotlib inline

## 1. Data Generation (Mock)
Since we don't have the live dataset, we'll generate a mock dataset that matches the schema.

In [ ]:
def generate_mock_data(n_samples=1000, filepath='train.csv'):
    np.random.seed(42)
    
    data = {
        'customer_id': [f'cust_{i:05d}' for i in range(n_samples)],
        'avg_spend': np.random.uniform(10, 250, n_samples),
        'avg_items_per_order': np.random.uniform(1, 15, n_samples),
        'days_since_last_visit': np.random.uniform(0, 90, n_samples),
        'preferred_category': np.random.choice(['electronics', 'apparel', 'home_goods'], n_samples),
        'uses_coupons': np.random.choice([0, 1], n_samples),
        'total_sessions': np.random.randint(1, 150, n_samples),
        'avg_session_duration': np.random.uniform(1, 45, n_samples)
    }
    
    df = pd.DataFrame(data)
    df.to_csv(filepath, index=False)
    print(f"Generated mock data at {filepath}")

# Check if train.csv exists, if not create it
if not os.path.exists('train.csv'):
    generate_mock_data()

## 2. Preprocessing
We load the data, handle missing values, and scale/encode features.

In [ ]:
def load_and_preprocess(filepath):
    # Load
    df = pd.read_csv(filepath)
    print(f"Data loaded. Shape: {df.shape}")
    
    # Clean
    if df.isnull().sum().sum() > 0:
        df = df.dropna()
    if df.duplicated().sum() > 0:
        df = df.drop_duplicates()
        
    # Define features
    numeric_features = ['avg_spend', 'avg_items_per_order', 'days_since_last_visit', 
                        'total_sessions', 'avg_session_duration']
    categorical_features = ['preferred_category']
    boolean_features = ['uses_coupons']
    
    # Pipeline
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numeric_features + boolean_features), 
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
        ])
    
    # Transform
    X = preprocessor.fit_transform(df.drop(columns=['customer_id']))
    return df, X, preprocessor

df, X_processed, preprocessor = load_and_preprocess('train.csv')

## 3. Clustering Analysis (The Elbow Method)
We'll test k from 2 to 10 to find the optimal number of clusters.

In [ ]:
inertias = []
silhouette_scores = []
k_range = range(2, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_processed)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_processed, kmeans.labels_))

fig, ax1 = plt.subplots(figsize=(10, 6))

color = 'tab:red'
ax1.set_xlabel('Number of clusters (k)')
ax1.set_ylabel('Inertia', color=color)
ax1.plot(k_range, inertias, marker='o', color=color)
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()  
color = 'tab:blue'
ax2.set_ylabel('Silhouette Score', color=color)
ax2.plot(k_range, silhouette_scores, marker='s', color=color)
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Elbow Method & Silhouette Score')
plt.show()

## 4. Final Model Training & Prediction
Based on the analysis, we'll choose a K (e.g., 5) and generate the submission.

In [ ]:
n_clusters = 5
model = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
model.fit(X_processed)
labels = model.predict(X_processed)

# Add to dataframe
df['cluster'] = labels

print("Cluster Distribution:")
print(df['cluster'].value_counts().sort_index())

## 5. Save Submission
Generate the CSV file required for the competition.

In [ ]:
submission = df[['customer_id', 'cluster']]
submission.to_csv('submission.csv', index=False)
print("Submission saved to submission.csv")
submission.head()